In [1]:
# @title
%%javascript
%%writefile gpu_bubble_sort.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

/* start GPU Automation functions*/
void dataTransfer(int *to, int *from, size_t length, cudaMemcpyKind direction){
    cudaError_t error;
    error = cudaMemcpy(to, from, length * sizeof(int), (cudaMemcpyKind)direction);
    if(error != cudaSuccess){
        printf("dataTransfer error: %s\n", cudaGetErrorString(error));
        exit(2);
    }
}

void gpuMalloc(int **d_g, size_t length){
    cudaError_t error;
    error = cudaMalloc ((void **) d_g, length * sizeof(int));
    if(error != cudaSuccess){
            printf("gpuMalloc error: %s\n", cudaGetErrorString(error));
            exit(1);
    }
}

void gpuFree(int *d_g){
    cudaError_t error;
    error = cudaFree(d_g);
    if(error != cudaSuccess){
        printf("gpuFree error: %s\n", cudaGetErrorString(error));
        exit(3);
    }

}

void synchronizeKernel(){
    cudaError_t error = cudaGetLastError();
    if(error != cudaSuccess){
        printf("Kernel Launch Failed\n");
        exit(4);
    }
}
/* end GPU Automation functions*/

//Kernel Function
__global__ void bubble(int *array, size_t length, int shifted){
    unsigned int x = (blockIdx.x * blockDim.x + threadIdx.x)*2 + shifted;
    unsigned int y = (blockIdx.x * blockDim.x + threadIdx.x)*2 + 1 + shifted;

    if(x < length && y < length){
      if(array[x] > array[y]){
        int s = array[x];
        array[x] = array[y];
        array[y] = s;
      }
    }

}
//Marshalling Function
void prepare_bubble_sort(int *array, size_t length){
  int *gpu_array;
  gpuMalloc(&gpu_array, length);

  dataTransfer(gpu_array, array, length, cudaMemcpyHostToDevice);


  int blockDim = 256;
  int gridDim = ((length / 2) + blockDim - 1) / blockDim;

  int loop_max = (int)(length/2) + 1;
  for(int i = 0; i < loop_max; i++){
    bubble <<< blockDim, gridDim>>> (gpu_array, length, 0);
    synchronizeKernel();

    cudaDeviceSynchronize();
    bubble <<<gridDim, blockDim>>> (gpu_array, length, 1);

    synchronizeKernel();
    cudaDeviceSynchronize();
  }

  dataTransfer(array, gpu_array, length, cudaMemcpyDeviceToHost);
  gpuFree(gpu_array);
}

//Main obviously
int main(int argc, char **argv){

  //Default values
  int length_temp = 0;
  int modulus = 65536;


  //Read from command line arguments
  if(argc < 2 || argc > 3){
    printf("Please provide an argument for the length of the array to sort\n");
    exit(1);
  }
  if(argc == 2 || argc == 3){
    length_temp = atoi(argv[1]);
    if(length_temp == 0){
      printf("Error: Cannot sort an empty array!\n");
      exit(2);
    }else if(length_temp == 1){
      printf("Error: Cannot sort an single element array!\n");
      exit(3);
    }
  }
  if(argc == 3){
    modulus = atoi(argv[2]);
    if(modulus == 0){
      printf("Error: Bad max_length value\n");
      exit(4);
    }
  }

  //Cast length to constant for array sizing.
  const int length = length_temp;
  int array[length] = {0};


  //Load random data
  printf("Before Sorting\n");
  for(int i = 0; i < length; i++){
    array[i] = rand() % modulus;

    printf("\t%d ", array[i]);
    if((i+1) % 20 == 0){
      printf("\n");
    }
  }


  //Do GPU stuff
  prepare_bubble_sort(array, (size_t)length);


  //Displays "sorted" array and verifys its correct
  int check = array[0];
  printf("\n\nAfter Sorting\n");
  for(int i = 0; i < length; i++){
    printf("\t%d ", array[i]);
    if((i+1) % 20 == 0){
      printf("\n");
    }

    if(check != -1 && array[i] < check){
      check = -1;
    }
  }

  //Prints if sorted array passed checks;
  if(check == -1){
    printf("\nFailed Sorting\n");
  }else{
    printf("\nSucessfully Sorted\n");
  }

  printf("\n");
  printf("\n");
  printf("\n");
  printf("\n");
  return 0;
}



Writing gpu_bubble_sort.cu


In [2]:
# @title
%%writefile makefile
CC = nvcc

# Compiler flags //unecessary compiler flags // -Wall: warnings -g: debug info
CFLAGS = -Xcudafe --diag_suppress=2464 -arch=sm_75

# Source files
SRCS = gpu_bubble_sort.cu

# Object files
OBJS = $(SRCS:.cu=.o)

# Executable name
TARGET = sort

# Default target
all: $(TARGET)

# Linking the executable // $@ sourcefile $^ all files/prerequesites
$(TARGET): $(OBJS)
	$(CC) $(CFLAGS) -o $@ $^

# Compiling .c to .o //$< sourcefile $@ generated file name
%.o: %.cu
	$(CC) $(CFLAGS) -c $< -o $@

# Clean up // build clean
clean:
	rm -f $(OBJS) $(TARGET)
.PHONY: all clean


Writing makefile


In [3]:
!make clean
!make

rm -f gpu_bubble_sort.o sort
nvcc -Xcudafe --diag_suppress=2464 -arch=sm_75 -c gpu_bubble_sort.cu -o gpu_bubble_sort.o
nvcc -Xcudafe --diag_suppress=2464 -arch=sm_75 -o sort gpu_bubble_sort.o


Command Syntax:
---
./sort array_length max_value

Required Parameters:
---

- array_length

non-Required Parameters:
---

- max_value defaults to 65536







In [4]:
!nvprof ./sort 65536 10000


Streaming output truncated to the last 5000 lines.
	4653 	250 	4066 	7833 	2169 	5593 	6611 	3198 	9469 	5744 	7869 	5791 	1196 	7978 	5409 	637 	4438 	5511 	91 	7255 
	5473 	659 	4224 	1482 	3670 	5340 	9721 	9220 	404 	6360 	9496 	5057 	2962 	3563 	9242 	5131 	5508 	5854 	4682 	1329 
	7950 	8903 	7120 	9146 	6881 	8881 	6136 	7671 	745 	6227 	4927 	6218 	3238 	9151 	4052 	6908 	843 	3773 	2480 	1247 
	6485 	1977 	2657 	9448 	1892 	8251 	931 	7400 	4105 	5613 	5081 	8407 	4516 	2201 	7554 	7749 	7435 	3690 	5421 	8180 
	6269 	348 	4398 	5859 	5851 	8451 	2767 	6694 	8576 	1599 	4294 	1414 	3576 	3303 	7214 	5468 	1554 	8145 	9220 	2012 
	111 	4301 	419 	979 	2855 	4325 	8729 	290 	4367 	4150 	8470 	6988 	850 	9220 	2847 	6701 	7671 	1966 	9747 	2600 
	3566 	4041 	4014 	7142 	3696 	1228 	8963 	5251 	5725 	8183 	3615 	5836 	8837 	4034 	3168 	8044 	8360 	1897 	8334 	9079 
	2399 	3156 	6068 	9601 	2376 	8915 	6302 	6400 	7234 	2401 	9000 	800 	2795 	3014 	4294 	6491 	594 	9609 	8094 	6319